In [ ]:
# ============================================================
#   AIR PASSENGER TIME SERIES ANALYSIS
#   Models: Moving Average, ARIMA, SARIMA, Prophet
#   Dataset: Kaggle Air Passenger Dataset
# ============================================================

# ============================================================
# STEP 1: INSTALL LIBRARIES (Run this in Kaggle notebook cell)
# ============================================================
# !pip install prophet
# !pip install pmdarima

# ============================================================
# STEP 2: IMPORT ALL LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

# For ARIMA & SARIMA
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller          # ADF test for stationarity
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf  # ACF & PACF plots
import pmdarima as pm                                   # auto_arima

# For Prophet
from prophet import Prophet

# For evaluation metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("✅ All libraries imported successfully!")


# ============================================================
# STEP 3: LOAD AND EXPLORE DATA
# ============================================================

# Load the dataset
# On Kaggle, the file path will be: /kaggle/input/air-passenger-data-for-time-series-analysis/AirPassengers.csv
df = pd.read_csv('AirPassengers.csv')

print("\n📊 First 10 rows of data:")
print(df.head(10))

print("\n📋 Dataset Info:")
print(df.info())

print("\n📈 Basic Statistics:")
print(df.describe())

print(f"\n📅 Date range: {df['Month'].min()} to {df['Month'].max()}")
print(f"📦 Total rows: {len(df)}")


# ============================================================
# STEP 4: DATA PREPROCESSING
# ============================================================

# Convert 'Month' column to datetime format
df['Month'] = pd.to_datetime(df['Month'])

# Set Month as index (important for time series!)
df.set_index('Month', inplace=True)

# Rename column for clarity
df.columns = ['Passengers']

print("\n✅ Data after preprocessing:")
print(df.head())
print(f"\nIndex type: {type(df.index)}")

# Set frequency to Monthly Start
df = df.asfreq('MS')


# ============================================================
# STEP 5: VISUALIZE THE DATA
# ============================================================

fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Plot 1: Original data
axes[0].plot(df.index, df['Passengers'], color='steelblue', linewidth=2)
axes[0].set_title('✈️ Air Passengers Over Time (1949–1960)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Passengers')
axes[0].set_xlabel('Year')
axes[0].grid(True, alpha=0.3)

# Plot 2: After 1st differencing (to show stationarity concept)
df_diff = df['Passengers'].diff().dropna()
axes[1].plot(df_diff.index, df_diff.values, color='darkorange', linewidth=2)
axes[1].set_title('📉 After 1st Differencing (Removing Trend)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Differenced Values')
axes[1].set_xlabel('Year')
axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[1].grid(True, alpha=0.3)

# Plot 3: Seasonal pattern (average passengers per month)
monthly_avg = df.groupby(df.index.month)['Passengers'].mean()
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
axes[2].bar(month_names, monthly_avg.values, color='mediumseagreen', edgecolor='black')
axes[2].set_title('📅 Average Passengers by Month (Seasonality Pattern)', fontsize=14, fontweight='bold')
axes[2].set_ylabel('Average Passengers')
axes[2].set_xlabel('Month')
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('01_data_exploration.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Plot saved: 01_data_exploration.png")


# ============================================================
# STEP 6: CHECK STATIONARITY — ADF TEST
# ============================================================

print("\n" + "="*55)
print("📊 STATIONARITY CHECK — ADF TEST")
print("="*55)

def check_stationarity(series, name="Series"):
    """
    ADF Test: 
    H0 (Null Hypothesis)  = Data is NOT stationary
    H1 (Alt  Hypothesis)  = Data IS stationary
    
    If p-value < 0.05 → Reject H0 → Data IS stationary ✅
    If p-value > 0.05 → Fail to reject H0 → Data NOT stationary ❌
    """
    result = adfuller(series.dropna())
    print(f"\n🔍 Testing: {name}")
    print(f"   ADF Statistic : {result[0]:.4f}")
    print(f"   p-value       : {result[1]:.4f}")
    print(f"   Critical Values:")
    for key, val in result[4].items():
        print(f"      {key}: {val:.4f}")
    
    if result[1] < 0.05:
        print(f"   ✅ RESULT: Data IS Stationary (p < 0.05)")
    else:
        print(f"   ❌ RESULT: Data is NOT Stationary (p > 0.05) → Need Differencing")
    return result[1]

# Test original data
p_original = check_stationarity(df['Passengers'], "Original Passengers Data")

# Test after 1st differencing
p_diff1 = check_stationarity(df['Passengers'].diff(), "After 1st Differencing")

# Test after 2nd differencing if needed
if p_diff1 > 0.05:
    p_diff2 = check_stationarity(df['Passengers'].diff().diff(), "After 2nd Differencing")


# ============================================================
# STEP 7: ACF AND PACF PLOTS (to find p and q for ARIMA)
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# ACF of original data
plot_acf(df['Passengers'], lags=40, ax=axes[0,0], color='steelblue')
axes[0,0].set_title('ACF — Original Data')

# PACF of original data
plot_pacf(df['Passengers'], lags=40, ax=axes[0,1], color='steelblue')
axes[0,1].set_title('PACF — Original Data')

# ACF of differenced data
plot_acf(df['Passengers'].diff().dropna(), lags=40, ax=axes[1,0], color='darkorange')
axes[1,0].set_title('ACF — After 1st Differencing')

# PACF of differenced data
plot_pacf(df['Passengers'].diff().dropna(), lags=40, ax=axes[1,1], color='darkorange')
axes[1,1].set_title('PACF — After 1st Differencing')

plt.suptitle('ACF & PACF Plots to Determine p and q', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('02_acf_pacf.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Plot saved: 02_acf_pacf.png")


# ============================================================
# STEP 8: TRAIN-TEST SPLIT
# ============================================================

# Use last 24 months (2 years) for testing
# Everything before that for training

train_size = len(df) - 24
train = df.iloc[:train_size]
test  = df.iloc[train_size:]

print(f"\n📦 Train size: {len(train)} months ({train.index[0].date()} to {train.index[-1].date()})")
print(f"📦 Test  size: {len(test)}  months ({test.index[0].date()}  to {test.index[-1].date()})")


# ============================================================
# MODEL 1: MOVING AVERAGE
# ============================================================

print("\n" + "="*55)
print("📊 MODEL 1: MOVING AVERAGE")
print("="*55)

def moving_average_forecast(train, test, window=12):
    """
    Simple Moving Average:
    - Calculate rolling mean with given window
    - Use last window average as forecast for all future steps
    
    Window=12 means average of last 12 months
    """
    # Calculate rolling mean on training data
    rolling = train['Passengers'].rolling(window=window)
    
    # The forecast = last computed rolling mean value repeated for all test periods
    last_ma_value = rolling.mean().iloc[-1]
    
    # Create forecast series
    ma_forecast = pd.Series([last_ma_value] * len(test), index=test.index)
    
    return ma_forecast, rolling.mean()

# Run Moving Average with window=12 (yearly)
ma_forecast, ma_rolling = moving_average_forecast(train, test, window=12)

# Calculate errors
ma_mae  = mean_absolute_error(test['Passengers'], ma_forecast)
ma_rmse = np.sqrt(mean_squared_error(test['Passengers'], ma_forecast))

print(f"\n📉 Moving Average Results (window=12):")
print(f"   MAE  : {ma_mae:.2f}  (average error in passengers)")
print(f"   RMSE : {ma_rmse:.2f} (root mean square error)")

# Plot
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(train.index, train['Passengers'], label='Training Data', color='steelblue', linewidth=2)
ax.plot(test.index,  test['Passengers'],  label='Actual Test Data', color='green', linewidth=2)
ax.plot(train.index, ma_rolling, label='Rolling Mean (train)', color='orange', linewidth=1.5, linestyle='--')
ax.plot(test.index,  ma_forecast, label=f'MA Forecast (window=12)', color='red', linewidth=2, linestyle='--')
ax.fill_between(test.index, ma_forecast * 0.9, ma_forecast * 1.1, alpha=0.2, color='red', label='±10% band')
ax.set_title('✈️ Moving Average Forecast', fontsize=14, fontweight='bold')
ax.set_ylabel('Passengers')
ax.set_xlabel('Year')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('03_moving_average.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Plot saved: 03_moving_average.png")


# ============================================================
# MODEL 2: ARIMA
# ============================================================

print("\n" + "="*55)
print("📊 MODEL 2: ARIMA")
print("="*55)

# --- Method A: Auto ARIMA (finds best p,d,q automatically) ---
print("\n🤖 Running auto_arima to find best (p,d,q)...")
print("   (This may take 30-60 seconds...)")

auto_model = pm.auto_arima(
    train['Passengers'],
    start_p=0, max_p=5,      # search p from 0 to 5
    start_q=0, max_q=5,      # search q from 0 to 5
    d=None,                   # auto-detect d using ADF test
    seasonal=False,           # no seasonality for plain ARIMA
    stepwise=True,            # faster search
    information_criterion='aic',  # use AIC to compare models
    trace=True,               # print results
    error_action='ignore',
    suppress_warnings=True
)

print(f"\n✅ Best ARIMA order found: {auto_model.order}")
print(f"   AIC Score: {auto_model.aic():.2f}")
print("\n📋 Model Summary:")
print(auto_model.summary())

# --- Method B: Manual ARIMA (using the order found above) ---
p, d, q = auto_model.order

arima_model = ARIMA(train['Passengers'], order=(p, d, q))
arima_result = arima_model.fit()

# Forecast for test period
arima_forecast_obj = arima_result.get_forecast(steps=len(test))
arima_forecast     = arima_forecast_obj.predicted_mean
arima_conf_int     = arima_forecast_obj.conf_int()  # confidence interval

# Align index
arima_forecast.index   = test.index
arima_conf_int.index   = test.index

# Calculate errors
arima_mae  = mean_absolute_error(test['Passengers'], arima_forecast)
arima_rmse = np.sqrt(mean_squared_error(test['Passengers'], arima_forecast))

print(f"\n📉 ARIMA{auto_model.order} Results:")
print(f"   MAE  : {arima_mae:.2f}")
print(f"   RMSE : {arima_rmse:.2f}")

# Plot
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(train.index, train['Passengers'], label='Training Data', color='steelblue', linewidth=2)
ax.plot(test.index,  test['Passengers'],  label='Actual Test Data', color='green', linewidth=2)
ax.plot(test.index,  arima_forecast,      label=f'ARIMA{auto_model.order} Forecast', color='red', linewidth=2)
ax.fill_between(test.index,
                arima_conf_int.iloc[:, 0],
                arima_conf_int.iloc[:, 1],
                alpha=0.2, color='red', label='95% Confidence Interval')
ax.set_title(f'✈️ ARIMA{auto_model.order} Forecast', fontsize=14, fontweight='bold')
ax.set_ylabel('Passengers')
ax.set_xlabel('Year')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('04_arima.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Plot saved: 04_arima.png")


# ============================================================
# MODEL 3: SARIMA
# ============================================================

print("\n" + "="*55)
print("📊 MODEL 3: SARIMA")
print("="*55)

# --- Auto SARIMA (finds best p,d,q,P,D,Q automatically) ---
print("\n🤖 Running auto_arima with seasonality (SARIMA)...")
print("   (This may take 1-2 minutes...)")

sarima_auto = pm.auto_arima(
    train['Passengers'],
    start_p=0, max_p=3,
    start_q=0, max_q=3,
    d=None,
    start_P=0, max_P=2,      # seasonal AR terms
    start_Q=0, max_Q=2,      # seasonal MA terms
    D=None,                   # auto-detect seasonal differencing
    seasonal=True,            # ✅ enable seasonality
    m=12,                     # m=12 for monthly data with yearly seasonality
    stepwise=True,
    information_criterion='aic',
    trace=True,
    error_action='ignore',
    suppress_warnings=True
)

print(f"\n✅ Best SARIMA order found:")
print(f"   Non-seasonal: {sarima_auto.order}")
print(f"   Seasonal    : {sarima_auto.seasonal_order}")
print(f"   AIC Score   : {sarima_auto.aic():.2f}")

# Fit SARIMA model
sarima_model = SARIMAX(
    train['Passengers'],
    order=sarima_auto.order,
    seasonal_order=sarima_auto.seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False
)
sarima_result = sarima_model.fit(disp=False)

print("\n📋 SARIMA Model Summary:")
print(sarima_result.summary())

# Forecast
sarima_forecast_obj = sarima_result.get_forecast(steps=len(test))
sarima_forecast     = sarima_forecast_obj.predicted_mean
sarima_conf_int     = sarima_forecast_obj.conf_int()

sarima_forecast.index = test.index
sarima_conf_int.index = test.index

# Calculate errors
sarima_mae  = mean_absolute_error(test['Passengers'], sarima_forecast)
sarima_rmse = np.sqrt(mean_squared_error(test['Passengers'], sarima_forecast))

print(f"\n📉 SARIMA Results:")
print(f"   MAE  : {sarima_mae:.2f}")
print(f"   RMSE : {sarima_rmse:.2f}")

# Plot
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(train.index, train['Passengers'], label='Training Data', color='steelblue', linewidth=2)
ax.plot(test.index,  test['Passengers'],  label='Actual Test Data', color='green', linewidth=2)
ax.plot(test.index,  sarima_forecast,     label=f'SARIMA Forecast', color='purple', linewidth=2)
ax.fill_between(test.index,
                sarima_conf_int.iloc[:, 0],
                sarima_conf_int.iloc[:, 1],
                alpha=0.2, color='purple', label='95% Confidence Interval')
ax.set_title(f'✈️ SARIMA{sarima_auto.order}×{sarima_auto.seasonal_order} Forecast',
             fontsize=14, fontweight='bold')
ax.set_ylabel('Passengers')
ax.set_xlabel('Year')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('05_sarima.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Plot saved: 05_sarima.png")

# Plot residuals (errors) to check model quality
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
sarima_result.plot_diagnostics(fig=fig)
plt.suptitle('SARIMA Residual Diagnostics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('05b_sarima_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Plot saved: 05b_sarima_diagnostics.png")


# ============================================================
# MODEL 4: PROPHET
# ============================================================

print("\n" + "="*55)
print("📊 MODEL 4: PROPHET")
print("="*55)

# Prophet needs data in specific format:
# Column 'ds' = dates
# Column 'y'  = values

train_prophet = train.reset_index().rename(columns={'Month': 'ds', 'Passengers': 'y'})
test_prophet  = test.reset_index().rename(columns={'Month': 'ds', 'Passengers': 'y'})

print("\nProphet training data format:")
print(train_prophet.head())

# Build Prophet model
prophet_model = Prophet(
    seasonality_mode='multiplicative',  # ✅ important for air passenger data!
    yearly_seasonality=True,
    weekly_seasonality=False,    # monthly data, no weekly pattern
    daily_seasonality=False,     # monthly data, no daily pattern
    changepoint_prior_scale=0.05,  # flexibility of trend (higher = more flexible)
    seasonality_prior_scale=10     # strength of seasonality
)

# Fit the model
print("\n🤖 Fitting Prophet model...")
prophet_model.fit(train_prophet)
print("✅ Prophet model fitted!")

# Create future dataframe for forecasting
# We tell Prophet how many months ahead to predict
future = prophet_model.make_future_dataframe(
    periods=len(test),
    freq='MS'           # MS = Month Start
)

print(f"\nFuture dataframe shape: {future.shape}")
print("Last few dates in future:")
print(future.tail())

# Make predictions
forecast = prophet_model.predict(future)

print("\n📋 Forecast columns explanation:")
print("   ds          = date")
print("   yhat        = predicted value")
print("   yhat_lower  = lower bound of prediction")
print("   yhat_upper  = upper bound of prediction")
print("   trend       = trend component")
print("   yearly      = seasonal component")

# Extract only test period predictions
prophet_test_forecast = forecast[forecast['ds'].isin(test_prophet['ds'])]

# Calculate errors
prophet_mae  = mean_absolute_error(test_prophet['y'], prophet_test_forecast['yhat'])
prophet_rmse = np.sqrt(mean_squared_error(test_prophet['y'], prophet_test_forecast['yhat']))

print(f"\n📉 Prophet Results:")
print(f"   MAE  : {prophet_mae:.2f}")
print(f"   RMSE : {prophet_rmse:.2f}")

# Plot 1: Prophet's built-in plot
fig = prophet_model.plot(forecast, figsize=(14, 6))
plt.title('✈️ Prophet Forecast', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('06_prophet_forecast.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Plot saved: 06_prophet_forecast.png")

# Plot 2: Prophet's component decomposition (Trend + Seasonality)
fig = prophet_model.plot_components(forecast, figsize=(14, 8))
plt.suptitle('Prophet Components: Trend & Seasonality', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('07_prophet_components.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Plot saved: 07_prophet_components.png")

# Plot 3: Custom plot showing train, actual test, prophet forecast
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(train.index, train['Passengers'], label='Training Data', color='steelblue', linewidth=2)
ax.plot(test.index,  test['Passengers'],  label='Actual Test Data', color='green', linewidth=2)
ax.plot(pd.to_datetime(prophet_test_forecast['ds']),
        prophet_test_forecast['yhat'],
        label='Prophet Forecast', color='darkorange', linewidth=2)
ax.fill_between(pd.to_datetime(prophet_test_forecast['ds']),
                prophet_test_forecast['yhat_lower'],
                prophet_test_forecast['yhat_upper'],
                alpha=0.2, color='darkorange', label='Uncertainty Interval')
ax.set_title('✈️ Prophet Forecast vs Actual', fontsize=14, fontweight='bold')
ax.set_ylabel('Passengers')
ax.set_xlabel('Year')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('08_prophet_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Plot saved: 08_prophet_vs_actual.png")


# ============================================================
# STEP 9: MODEL COMPARISON
# ============================================================

print("\n" + "="*55)
print("📊 FINAL MODEL COMPARISON")
print("="*55)

results = pd.DataFrame({
    'Model': ['Moving Average', 'ARIMA', 'SARIMA', 'Prophet'],
    'MAE':   [round(ma_mae,2), round(arima_mae,2), round(sarima_mae,2), round(prophet_mae,2)],
    'RMSE':  [round(ma_rmse,2), round(arima_rmse,2), round(sarima_rmse,2), round(prophet_rmse,2)]
})

results['MAE_Rank']  = results['MAE'].rank().astype(int)
results['RMSE_Rank'] = results['RMSE'].rank().astype(int)
results['Overall_Rank'] = (results['MAE_Rank'] + results['RMSE_Rank']) / 2

results = results.sort_values('Overall_Rank')

print("\n" + results.to_string(index=False))
print(f"\n🏆 Best Model: {results.iloc[0]['Model']}")

# Plot comparison bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']

# MAE comparison
bars1 = axes[0].bar(results['Model'], results['MAE'], color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_title('MAE Comparison (Lower = Better)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Mean Absolute Error')
axes[0].set_xlabel('Model')
for bar, val in zip(bars1, results['MAE']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.1f}', ha='center', va='bottom', fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

# RMSE comparison
bars2 = axes[1].bar(results['Model'], results['RMSE'], color=colors, edgecolor='black', linewidth=0.5)
axes[1].set_title('RMSE Comparison (Lower = Better)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Root Mean Square Error')
axes[1].set_xlabel('Model')
for bar, val in zip(bars2, results['RMSE']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.1f}', ha='center', va='bottom', fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle('✈️ Model Performance Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('09_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Plot saved: 09_model_comparison.png")


# ============================================================
# STEP 10: ALL MODELS ON ONE PLOT
# ============================================================

fig, ax = plt.subplots(figsize=(16, 8))

# Training data
ax.plot(train.index, train['Passengers'],
        label='Training Data', color='steelblue', linewidth=2.5)

# Actual test data
ax.plot(test.index, test['Passengers'],
        label='Actual Test Data', color='black', linewidth=2.5, linestyle='-')

# Moving Average
ax.plot(test.index, ma_forecast,
        label=f'Moving Average (MAE={ma_mae:.1f})', color='gray',
        linewidth=2, linestyle='--')

# ARIMA
ax.plot(test.index, arima_forecast,
        label=f'ARIMA (MAE={arima_mae:.1f})', color='red',
        linewidth=2, linestyle='--')

# SARIMA
ax.plot(test.index, sarima_forecast,
        label=f'SARIMA (MAE={sarima_mae:.1f})', color='purple',
        linewidth=2, linestyle='--')

# Prophet
ax.plot(pd.to_datetime(prophet_test_forecast['ds']),
        prophet_test_forecast['yhat'],
        label=f'Prophet (MAE={prophet_mae:.1f})', color='darkorange',
        linewidth=2, linestyle='--')

# Vertical line separating train and test
ax.axvline(x=test.index[0], color='black', linestyle=':', linewidth=1.5, alpha=0.7)
ax.text(test.index[0], ax.get_ylim()[0],
        '  Train | Test', fontsize=10, color='black', alpha=0.7)

ax.set_title('✈️ All Models Comparison on Air Passenger Data', fontsize=15, fontweight='bold')
ax.set_ylabel('Number of Passengers')
ax.set_xlabel('Year')
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('10_all_models_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Plot saved: 10_all_models_comparison.png")


# ============================================================
# STEP 11: FUTURE FORECAST (Beyond the dataset)
# ============================================================

print("\n" + "="*55)
print("🔮 FUTURE FORECAST — Next 24 Months")
print("="*55)

# Use SARIMA (usually best for this dataset) for future forecast
# Refit on FULL data
full_sarima = SARIMAX(
    df['Passengers'],
    order=sarima_auto.order,
    seasonal_order=sarima_auto.seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

future_forecast     = full_sarima.get_forecast(steps=24)
future_pred         = future_forecast.predicted_mean
future_conf         = future_forecast.conf_int()

print("\n📅 Future Predictions (next 24 months):")
print(future_pred.to_frame('Predicted Passengers').round(0).to_string())

# Plot future forecast
fig, ax = plt.subplots(figsize=(16, 7))

ax.plot(df.index, df['Passengers'],
        label='Historical Data', color='steelblue', linewidth=2.5)
ax.plot(future_pred.index, future_pred,
        label='SARIMA Future Forecast', color='purple', linewidth=2.5, linestyle='--')
ax.fill_between(future_pred.index,
                future_conf.iloc[:, 0],
                future_conf.iloc[:, 1],
                alpha=0.25, color='purple', label='95% Confidence Interval')

ax.axvline(x=df.index[-1], color='black', linestyle=':', linewidth=1.5, alpha=0.7)
ax.set_title('✈️ SARIMA Future Forecast — Beyond 1960', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Passengers')
ax.set_xlabel('Year')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('11_future_forecast.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Plot saved: 11_future_forecast.png")


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "="*55)
print("🎯 COMPLETE SUMMARY")
print("="*55)
print("""
📚 What we did:
   1. Loaded and explored Air Passenger dataset
   2. Checked stationarity using ADF Test
   3. Plotted ACF & PACF to understand patterns
   4. Split data into train (80%) and test (20%)
   5. Built 4 models: MA, ARIMA, SARIMA, Prophet
   6. Compared all models using MAE and RMSE
   7. Forecasted future values using best model

📊 Models explained:
   • Moving Average → Simple, uses average of last N values
   • ARIMA(p,d,q)   → Uses past values + errors, no seasonality
   • SARIMA         → ARIMA + seasonal patterns
   • Prophet        → Facebook model, handles trend+seasonality+holidays

🏆 For Air Passenger Data:
   • SARIMA and Prophet are typically best
   • Because data has both TREND and SEASONALITY
   • Prophet is easiest to use
   • SARIMA gives more control and interpretability

📁 Output files saved:
   01_data_exploration.png
   02_acf_pacf.png
   03_moving_average.png
   04_arima.png
   05_sarima.png
   05b_sarima_diagnostics.png
   06_prophet_forecast.png
   07_prophet_components.png
   08_prophet_vs_actual.png
   09_model_comparison.png
   10_all_models_comparison.png
   11_future_forecast.png
""")